<a href="https://colab.research.google.com/github/AnjanPayra/Ortho_Sim_Loc-essential-protein-prediction-using-orthology-and-priority-based-similarity-approach/blob/main/Ortho_Sim_Loc_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import os
import csv
import time
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, mean_squared_error, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)

warnings.filterwarnings("ignore")
np.random.seed(42)

UPLOAD_DIR = "/content"
OUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

DATASETS = ["YDIP", "YHQ", "YMBD", "YMIPS"]
PRIORITY_COMPARTMENTS = {"endoplasmic reticulum", "nucleus"}
ALPHA = 0.5          # weight between Sim_Loc and Orthologous score
K_SIGMA = 1           # K in the 3-point (K-sigma) threshold rule

In [28]:
# ======================================================================
# 0. Ground truth + genome-wide (network independent) reference tables
# ======================================================================
def load_essential_set():
    e = pd.read_excel(os.path.join(UPLOAD_DIR, "Essential.xlsx"), header=None)
    return set(e[0].astype(str).str.strip())


COMPARTMENT_KEYWORDS = [
    "nucleus", "nucleolus", "cytoplasm", "cytosol", "mitochondri",
    "golgi", "endoplasmic reticulum", "vacuole", "bud", "cell membrane",
    "membrane", "peroxisome", "cell wall", "ribosome", "endosome",
    "chromosome", "spindle", "nuclear", "secreted", "lipid particle"
]

def load_subcellular_map():
    sc = pd.read_csv(os.path.join(UPLOAD_DIR, "Sub_cellular.csv"))
    sc.columns = ["gene", "location_text"]
    sc = sc.dropna(subset=["gene"])
    sl_map = {}
    for _, row in sc.iterrows():
        gene = str(row["gene"]).strip()
        text = str(row["location_text"]).lower()
        found = {kw for kw in COMPARTMENT_KEYWORDS if kw in text}
        if found:
            sl_map[gene] = found
    return sl_map


def load_go_scores(dataset):
    """Functional activity score GO_Nb(u), precomputed per dataset."""
    path = os.path.join(UPLOAD_DIR, f"{dataset}_GO_final.csv")
    go = pd.read_csv(path, header=None, names=["protein", "GO_Nb"])
    go["protein"] = go["protein"].astype(str).str.strip()
    return dict(zip(go["protein"], go["GO_Nb"]))


def load_physico_chemical():
    """Genome-wide physico-chemical property table (protein-id indexed)."""
    # Changed from pd.read_csv to pd.read_excel and filename to All_Physicochemical.xlsx
    p = pd.read_excel(os.path.join(UPLOAD_DIR, "All_Physicochemical.xlsx"), header=None)
    p.columns = ["protein"] + [f"phys_{i}" for i in range(1, p.shape[1])]
    p["protein"] = p["protein"].astype(str).str.strip()
    p = p.set_index("protein")
    return p


def load_cog_annotation():
    """Genome-wide protein -> set(COG ids) mapping."""
    path = os.path.join(UPLOAD_DIR, "YDIP_level1_COG.csv")
    cog_map = {}
    with open(path) as f:
        for line in f:
            parts = [x.strip() for x in line.strip().split(",") if x.strip()]
            if not parts:
                continue
            protein, cogs = parts[0], parts[1:]
            cog_map[protein] = set(cogs)
    return cog_map


def load_cog_pair_scores():
    """COG-COG association score lookup, from StringDB COG_11.csv."""
    print("Loading COG-COG association scores (COG_11.csv) ...")
    t0 = time.time()
    pair_score = {}
    with open(os.path.join(UPLOAD_DIR, "COG_11.csv")) as f:
        r = csv.reader(f)
        for row in r:
            if len(row) < 3:
                continue
            c1, c2, score = row[0], row[1], float(row[2])
            pair_score[(c1, c2)] = score
            pair_score[(c2, c1)] = score
    print(f"  {len(pair_score)} directed COG pairs loaded in {time.time()-t0:.1f}s")
    return pair_score

In [16]:
# ======================================================================
# 1. Network loading
# ======================================================================
def load_network(dataset):
    path = os.path.join(UPLOAD_DIR, f"{dataset}.txt")
    df = pd.read_csv(path)
    df.columns = ["p1", "p2"]
    df = df.dropna()
    df = df[df["p1"].astype(str).str.strip() != df["p2"].astype(str).str.strip()]
    G = nx.Graph()
    G.add_edges_from(zip(df["p1"].astype(str).str.strip(), df["p2"].astype(str).str.strip()))
    G.remove_edges_from(nx.selfloop_edges(G))
    return G

In [17]:

# ======================================================================
# Algorithm 2 - step 1.1 : Centrality measures
# ======================================================================
def compute_centralities(G):
    deg = dict(G.degree())
    n = G.number_of_nodes()
    k_sample = min(200, n)
    betw = nx.betweenness_centrality(G, k=k_sample, seed=42)
    clus = nx.clustering(G)
    try:
        eig = nx.eigenvector_centrality(G, max_iter=500, tol=1e-04)
    except nx.PowerIterationFailedConvergence:
        eig = {n_: 0.0 for n_ in G.nodes()}
    close = nx.closeness_centrality(G)
    return {
        "degree": deg,
        "betweenness": betw,
        "closeness": close,
        "eigenvector": eig,
        "clustering": clus,
    }



In [18]:
# ======================================================================
# Algorithm 3 - step 5 : COG association / Orthologous score OS(p_i)
# ======================================================================
def compute_orthologous_scores(G, cog_map, pair_score):
    """
    A(u,v) = best (max) COG-COG association score between the COG sets
             annotating proteins u and v (StringDB-style ortholog score).
    OS(p_i) = sum_j A(p_i, j) / max_j A(p_i, j),  j in level-1 neighbours.
    """
    os_scores = {}
    edge_assoc = {}
    for u, v in G.edges():
        cu, cv = cog_map.get(u, set()), cog_map.get(v, set())
        best = 0.0
        if cu and cv:
            for c1 in cu:
                for c2 in cv:
                    s = pair_score.get((c1, c2))
                    if s is not None and s > best:
                        best = s
        edge_assoc[(u, v)] = best
        edge_assoc[(v, u)] = best

    for p in G.nodes():
        neigh_scores = [edge_assoc.get((p, t), 0.0) for t in G.neighbors(p)]
        neigh_scores = [s for s in neigh_scores if s > 0]
        if neigh_scores:
            os_scores[p] = sum(neigh_scores) / max(neigh_scores)
        else:
            os_scores[p] = 0.0
    return os_scores



In [19]:
# ======================================================================
# Algorithm 1 - step 4.1 : Subcellular membership  P_Ci  (Sl_memb)
#   "Proteins locating in Endoplasmic reticulum and Nucleus tend to
#    process indispensable functions"
# ======================================================================
def compute_subcellular_membership(nodes, sl_map):
    p_ci = {}
    for p in nodes:
        comps = sl_map.get(p, set())
        if comps:
            n_priority = len(comps & PRIORITY_COMPARTMENTS)
            p_ci[p] = n_priority / len(comps)
        else:
            p_ci[p] = 0.0
    return p_ci

In [20]:
# ======================================================================
# Algorithm 2 - steps 2 & 3 : classify essential-properties, cluster,
#                              Priority Fun_GO  N_c(i)
# ======================================================================
def build_priority_fun_go(df, feature_cols, go_scores):
    """
    2.1  classify centrality + physico-chemical values with a classifier
    2.2  keep membership values of properties predicted essential only
    3.1  K-means clustering on that subset (K chosen via silhouette)
    3.2  cluster-wise GO value (C_Go), threshold >= 5e-6 (GO_Nb already
         non-negative; proteins with GO_Nb below the threshold contribute 0)
    3.3  compute functionally-enriched Priority_FunGO N_c(i) per cluster
    """
    X = df[feature_cols].values
    y = df["essential"].values
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    clf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
    clf.fit(Xs, y)
    pred = clf.predict(Xs)

    essential_props = df.loc[pred == 1].copy()
    if len(essential_props) < 10:
        # fallback: not enough predicted-essential rows to cluster meaningfully
        essential_props = df.loc[df["essential"] == 1].copy()

    Xe = scaler.transform(essential_props[feature_cols].values)

    best_k, best_score, best_labels = None, -1, None
    validity_rows = []
    for k in range(2, 7):
        if len(Xe) <= k:
            continue
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(Xe)
        if len(set(labels)) < 2:
            continue
        sil = silhouette_score(Xe, labels)
        dbi = davies_bouldin_score(Xe, labels)
        ch = calinski_harabasz_score(Xe, labels)
        validity_rows.append({"K": k, "silhouette": sil, "davies_bouldin": dbi,
                               "calinski_harabasz": ch})
        if sil > best_score:
            best_score, best_k, best_labels = sil, k, labels

    validity_df = pd.DataFrame(validity_rows)

    if best_labels is None:
        # degenerate case: everything in one cluster
        best_k = 1
        best_labels = np.zeros(len(essential_props), dtype=int)

    essential_props["cluster"] = best_labels

    # GO-attribute value per cluster (threshold 5e-6 as in Fig.7 -> value
    # already >=0, floor tiny/NaN contributions to 0)
    GO_THRESH = 5e-6
    c_go = {}
    for c in sorted(essential_props["cluster"].unique()):
        members = essential_props.loc[essential_props["cluster"] == c, "protein"]
        vals = [go_scores.get(p, 0.0) for p in members]
        vals = [v for v in vals if v >= GO_THRESH]
        c_go[c] = float(np.mean(vals)) if vals else 0.0

    total = sum(c_go.values())
    n_c = {c: (v / total if total > 0 else 0.0) for c, v in c_go.items()}

    essential_props["N_c"] = essential_props["cluster"].map(n_c)
    priority_map = dict(zip(essential_props["protein"], essential_props["N_c"]))

    return priority_map, best_k, validity_df



In [21]:
# ======================================================================
# 3-sigma / K-sigma threshold rule (Fig.9 "3-point Threshold")
# ======================================================================
def k_sigma_threshold(values, K=K_SIGMA):
    values = np.asarray(values, dtype=float)
    alpha = values.mean()
    sigma = values.std()
    var = values.var()
    return alpha + K * sigma * (1 - 1 / (1 + var ** 2))

In [22]:
# ======================================================================
# Build the full per-protein table for one dataset
# ======================================================================
def build_dataset(dataset, essential_set, sl_map, cog_map, pair_score, phys_df):
    print(f"\n[{dataset}] loading network ...")
    G = load_network(dataset)
    print(f"  nodes={G.number_of_nodes()}  edges={G.number_of_edges()}")

    print(f"[{dataset}] computing centrality measures ...")
    cen = compute_centralities(G)

    print(f"[{dataset}] computing orthologous (COG) scores ...")
    os_scores = compute_orthologous_scores(G, cog_map, pair_score)

    print(f"[{dataset}] computing subcellular membership (P_Ci) ...")
    p_ci = compute_subcellular_membership(G.nodes(), sl_map)

    go_scores = load_go_scores(dataset)

    rows = []
    for n in G.nodes():
        phys = phys_df.loc[n] if n in phys_df.index else None
        row = {
            "protein": n,
            "degree": cen["degree"].get(n, 0),
            "betweenness": cen["betweenness"].get(n, 0.0),
            "closeness": cen["closeness"].get(n, 0.0),
            "eigenvector": cen["eigenvector"].get(n, 0.0),
            "clustering": cen["clustering"].get(n, 0.0),
            "OS": os_scores.get(n, 0.0),
            "P_Ci": p_ci.get(n, 0.0),
            "GO_Nb": go_scores.get(n, 0.0),
            "essential": 1 if n in essential_set else 0,
        }
        if phys is not None:
            for c in phys_df.columns:
                row[c] = phys[c]
        else:
            for c in phys_df.columns:
                row[c] = 0.0
        rows.append(row)

    df = pd.DataFrame(rows)

    # ---- Algorithm 2: Priority Fun_GO N_c(i) via classify + cluster ----
    centrality_cols = ["degree", "betweenness", "closeness", "eigenvector", "clustering"]
    phys_cols = list(phys_df.columns)
    feature_cols = centrality_cols + phys_cols

    priority_map, best_k, validity_df = build_priority_fun_go(df, feature_cols, go_scores)
    df["N_c"] = df["protein"].map(priority_map).fillna(0.0)

    # ---- Algorithm 1: Sim_Loc ----
    def sim_loc(row):
        nci, pci = row["N_c"], row["P_Ci"]
        denom = nci + pci
        return (nci * pci) / denom if denom > 0 else 0.0

    df["Sim_Loc"] = df.apply(sim_loc, axis=1)

    # ---- Algorithm 3: Ortho_Sim_Loc ----
    def ortho_sim_loc(row):
        sim, os_ = row["Sim_Loc"], row["OS"]
        denom = max(sim, os_)
        if denom == 0:
            return 0.0
        return (ALPHA * sim + (1 - ALPHA) * os_) / denom

    df["Ortho_Sim_Loc"] = df.apply(ortho_sim_loc, axis=1)

    return df, best_k, validity_df



In [23]:
# ======================================================================
# Evaluation: threshold-based prediction + continuous-score ROC/PR
# ======================================================================
def evaluate(df, dataset_name):
    y_true = df["essential"].values
    score = df["Ortho_Sim_Loc"].values

    fpr, tpr, roc_thresh = roc_curve(y_true, score)
    roc_auc = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(y_true, score)
    ap = average_precision_score(y_true, score)

    # ---- (a) Paper-defined 3-point / K-sigma threshold ----
    thresh_paper = k_sigma_threshold(score, K=K_SIGMA)
    y_pred_paper = (score >= thresh_paper).astype(int)
    acc_paper = accuracy_score(y_true, y_pred_paper)
    mse_paper = mean_squared_error(y_true, y_pred_paper)
    cm_paper = confusion_matrix(y_true, y_pred_paper)

    # ---- (b) Data-driven optimal threshold (Youden's J = TPR-FPR max) ----
    youden = tpr - fpr
    best_idx = int(np.argmax(youden))
    thresh_opt = roc_thresh[best_idx]
    y_pred_opt = (score >= thresh_opt).astype(int)
    acc_opt = accuracy_score(y_true, y_pred_opt)
    mse_opt = mean_squared_error(y_true, y_pred_opt)
    cm_opt = confusion_matrix(y_true, y_pred_opt)
    report_opt = classification_report(y_true, y_pred_opt, target_names=["non-essential", "essential"])

    print(f"\n[{dataset_name}] Ortho_Sim_Loc evaluation")
    print(f"  ROC-AUC  : {roc_auc:.4f}   PR-AUC : {ap:.4f}")
    print(f"  (a) Paper 3-sigma threshold = {thresh_paper:.4f} -> Accuracy={acc_paper:.4f}  MSE={mse_paper:.4f}")
    print(f"      Confusion matrix:\n{cm_paper}")
    print(f"  (b) Youden-optimal threshold = {thresh_opt:.4f} -> Accuracy={acc_opt:.4f}  MSE={mse_opt:.4f}")
    print(f"      Confusion matrix:\n{cm_opt}")
    print(report_opt)

    return {
        "dataset": dataset_name,
        "threshold_paper": thresh_paper, "accuracy_paper": acc_paper, "mse_paper": mse_paper,
        "cm_paper": cm_paper,
        "threshold_opt": thresh_opt, "accuracy_opt": acc_opt, "mse_opt": mse_opt,
        "cm_opt": cm_opt, "report_opt": report_opt,
        "roc_auc": roc_auc, "pr_auc": ap,
        "fpr": fpr, "tpr": tpr, "precision": prec, "recall": rec,
    }

In [29]:
# ======================================================================
# Main driver
# ======================================================================
def main():
    essential_set = load_essential_set()
    sl_map = load_subcellular_map()
    cog_map = load_cog_annotation()
    pair_score = load_cog_pair_scores()
    phys_df = load_physico_chemical()

    print(f"Essential reference proteins : {len(essential_set)}")
    print(f"Proteins with subcellular annotation : {len(sl_map)}")
    print(f"Proteins with COG annotation : {len(cog_map)}")
    print(f"Proteins with physico-chemical data : {len(phys_df)}")

    all_tables, all_results, k_summary = {}, {}, []

    for ds in DATASETS:
        df, best_k, validity_df = build_dataset(
            ds, essential_set, sl_map, cog_map, pair_score, phys_df
        )
        all_tables[ds] = df
        df.to_csv(os.path.join(OUT_DIR, f"{ds}_OrthoSimLoc_feature_table.csv"), index=False)
        validity_df.to_csv(os.path.join(OUT_DIR, f"{ds}_cluster_validity.csv"), index=False)
        k_summary.append({"dataset": ds, "best_K_clusters": best_k})

        result = evaluate(df, ds)
        all_results[ds] = result

    pd.DataFrame(k_summary).to_csv(os.path.join(OUT_DIR, "chosen_K_per_dataset.csv"), index=False)

    # -------------------- Summary metrics table --------------------
    summary = pd.DataFrame([{
        "Dataset": ds,
        "Nodes": len(all_tables[ds]),
        "Essential(labelled)": int(all_tables[ds]["essential"].sum()),
        "Threshold_paper(3sigma)": all_results[ds]["threshold_paper"],
        "Accuracy_paper": all_results[ds]["accuracy_paper"],
        "MSE_paper": all_results[ds]["mse_paper"],
        "Threshold_Youden": all_results[ds]["threshold_opt"],
        "Accuracy_Youden": all_results[ds]["accuracy_opt"],
        "MSE_Youden": all_results[ds]["mse_opt"],
        "ROC_AUC": all_results[ds]["roc_auc"],
        "PR_AUC": all_results[ds]["pr_auc"],
    } for ds in DATASETS])
    summary.to_csv(os.path.join(OUT_DIR, "OrthoSimLoc_summary_metrics.csv"), index=False)
    print("\n=== SUMMARY (Ortho_Sim_Loc) ===")
    print(summary.to_string(index=False))

    # -------------------- ROC curve --------------------
    plt.figure(figsize=(7, 6))
    for ds in DATASETS:
        r = all_results[ds]
        plt.plot(r["fpr"], r["tpr"], label=f"{ds} (AUC={r['roc_auc']:.3f})", linewidth=2)
    plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve — Essential Protein Prediction (Ortho_Sim_Loc)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "OrthoSimLoc_ROC_curves.png"), dpi=150)
    plt.close()

    # -------------------- PR curve --------------------
    plt.figure(figsize=(7, 6))
    for ds in DATASETS:
        r = all_results[ds]
        plt.plot(r["recall"], r["precision"], label=f"{ds} (AP={r['pr_auc']:.3f})", linewidth=2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve — Essential Protein Prediction (Ortho_Sim_Loc)")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "OrthoSimLoc_PR_curves.png"), dpi=150)
    plt.close()

    # -------------------- Bar chart --------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(DATASETS))
    width = 0.25
    ax.bar(x - width, summary["Accuracy_Youden"], width, label="Accuracy (Youden thresh.)")
    ax.bar(x, summary["ROC_AUC"], width, label="ROC-AUC")
    ax.bar(x + width, summary["PR_AUC"], width, label="PR-AUC")
    ax.set_xticks(x)
    ax.set_xticklabels(DATASETS)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("Ortho_Sim_Loc Model Performance Across Datasets")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "OrthoSimLoc_performance_comparison.png"), dpi=150)
    plt.close()

    print(f"\nAll outputs saved to {OUT_DIR}")
    return all_tables, all_results, summary


if __name__ == "__main__":
    main()

Loading COG-COG association scores (COG_11.csv) ...
  1916418 directed COG pairs loaded in 1.6s
Essential reference proteins : 1285
Proteins with subcellular annotation : 4081
Proteins with COG annotation : 5093
Proteins with physico-chemical data : 5094

[YDIP] loading network ...
  nodes=5093  edges=24743
[YDIP] computing centrality measures ...
[YDIP] computing orthologous (COG) scores ...
[YDIP] computing subcellular membership (P_Ci) ...

[YDIP] Ortho_Sim_Loc evaluation
  ROC-AUC  : 0.8035   PR-AUC : 0.6744
  (a) Paper 3-sigma threshold = 0.4108 -> Accuracy=0.3888  MSE=0.6112
      Confusion matrix:
[[ 871 3055]
 [  58 1109]]
  (b) Youden-optimal threshold = 0.5006 -> Accuracy=0.8989  MSE=0.1011
      Confusion matrix:
[[3926    0]
 [ 515  652]]
               precision    recall  f1-score   support

non-essential       0.88      1.00      0.94      3926
    essential       1.00      0.56      0.72      1167

     accuracy                           0.90      5093
    macro avg    